# Fitting

Use the **bao** kernel in your allocated HPC remote desktop. Edit these cells and the reusable functions in `pzbao/diagnostics/fitting.py`.

These small synthetic examples illustrate the interface; they are not BAO results. Scientific interpretation and acceptance remain yours.


In [ ]:
from pathlib import Path
import numpy as np
from pzbao.diagnostics import fitting as diagnostics
from pzbao.config import load_analysis_config
from pzbao.config.stages import stage_document
from pzbao.run import load_run, stage_units

RUN_ROOT = None  # Set to the named run directory you want to inspect.
run = load_run(RUN_ROOT) if RUN_ROOT is not None else None
config = run["config"] if run is not None else load_analysis_config()


## Settings and saved products

This is the section used by `bao fit`. Edit a complete stage YAML or a combined experiment YAML. CLI runs freeze scientific settings; use a new named run for a changed experiment.


In [ ]:
settings = stage_document(config, "fit")
settings


In [ ]:
if run is not None:
    units = stage_units("fit", config, run["metadata"]["workload"])
    products = list((run["root"] / "fit" / "units").glob("*/result.yaml"))
    print(f"{len(products)} saved units; {len(units)} configured units")


## Explore and plot

Replace the illustrative arrays with saved arrays from your chosen unit. Plotting functions accept an existing axis and return their data for inspection.


In [ ]:
s = np.linspace(50.0, 150.0, 20)
model = np.sin(s / 10.0) / s
data = model + np.linspace(-0.001, 0.001, len(s))
ax, values = diagnostics.plot_residuals(s, data, model, np.eye(len(s)) * 0.0001)
values["chi2"]


## Load one saved result

Set `LOAD_SAVED = True` after selecting your run and unit. These cells only read saved products; missing prerequisites remain visible as errors. The clustering example reads xi; P(k)-only runs can be inspected through the unit's `measurements.json` and named power NPZ files.


In [ ]:
LOAD_SAVED = False
SAVED_UNIT_INDEX = 0  # Choose the configured unit/route to inspect.
if LOAD_SAVED:
    if run is None:
        raise ValueError("Set RUN_ROOT before loading saved products")
    from pzbao.fitting import read_profiles

    unit = units[SAVED_UNIT_INDEX]
    directory = run["root"] / "fit" / "units" / unit["unit_id"]
    saved_profiles = read_profiles(directory / "profiles.npy")
    print(saved_profiles["parameters"])
    with np.load(directory / "fit_curve.npz", allow_pickle=False) as archive:
        saved_curve = {key: archive[key] for key in archive.files}
    from pzbao.figures.style import axes

    saved_ax = axes()
    saved_ax.plot(saved_curve["residual"] / saved_curve["standard_deviation"])
    saved_ax.set(xlabel="Multipole-major vector index", ylabel="Residual / marginal error")
    saved_values = saved_curve


## Interpretation

Record the chosen unit, comparisons, missing or failed products, relevant flags, and your scientific question here. Numerical stopping and report publication do not establish a validation verdict.
